# DASS — StyleGAN2-ADA + chọn ảnh sinh cho ảnh y tế mất cân bằng (chạy trên Colab)

Notebook này **không chứa logic**: mọi code nằm trong package `dass` (`src/`), mỗi cell gọi một lệnh CLI.
Mỗi lệnh là một tiến trình riêng (PyTorch và TensorFlow không tranh VRAM) và **chạy lại được**: artefact lưu
trên Drive, phần đã xong được khôi phục / bỏ qua.

`prepare → gan → sample → [fingerprint] → select → train → evaluate → report` — hoặc cả chuỗi: `dass … run`.

Runtime: **GPU** (A100 / L4 khuyến nghị; T4 chạy được nhưng train GAN chậm).

## 0. Kết nối Drive, lấy mã nguồn, cài đặt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os

# Nguồn mã — một trong ba cách:
# (a) VS Code + extension Colab: chọn src/, configs/, pyproject.toml, README.md -> chuột phải -> "Upload to Colab"
#     (file nằm ở /content, tự nhận diện).
# (b) Clone từ GitHub: điền REPO_URL.
# (c) Thư mục dự án trên Google Drive: PROJECT_ON_DRIVE.
REPO_URL = ""   # ví dụ "https://github.com/<user>/dass.git"
PROJECT_ON_DRIVE = "/content/drive/MyDrive/dass"

if os.path.exists("/content/pyproject.toml"):                       # (a)
    PROJECT_DIR = "/content"
    if os.path.isdir("/content/dass"):        # lỡ upload src/dass thay vì src/ -> đưa về đúng chỗ
        !mkdir -p /content/src/dass && rsync -a /content/dass/ /content/src/dass/ && rm -rf /content/dass
elif REPO_URL:                                                       # (b)
    PROJECT_DIR = "/content/dass-repo"
    if os.path.isdir(PROJECT_DIR):
        !git -C {PROJECT_DIR} pull -q
    else:
        !git clone -q {REPO_URL} {PROJECT_DIR}
else:                                                                # (c)
    PROJECT_DIR = "/content/dass-repo"
    !rsync -a --delete --exclude .git "{PROJECT_ON_DRIVE}/" {PROJECT_DIR}/

print("PROJECT_DIR =", PROJECT_DIR)
%cd {PROJECT_DIR}
# torch / tensorflow có sẵn trên Colab -> chỉ cài package + phụ thuộc nhẹ
!pip install -q -e . ninja click keras-hub

## 1. Chọn thực nghiệm

`EXPERIMENT`: `configs/experiments/isic2016_dass.yaml` hoặc `configs/experiments/brain_tumor_dass.yaml`.
`PROFILE = "configs/experiments/smoke.yaml"` để chạy thử nhanh (ghi vào thư mục `*_smoke`). Ghi đè thêm:
`EXTRA = "--set selection.gamma=0.25 --tag gamma025"`.

In [ ]:
EXPERIMENT = "configs/experiments/brain_tumor_dass.yaml"
PROFILE = ""          # "configs/experiments/smoke.yaml" để chạy thử
EXTRA = ""            # ví dụ "--set selection.gamma=0.25 --tag gamma025"
CFG = f"-c {EXPERIMENT}" + (f" -c {PROFILE}" if PROFILE else "") + (f" {EXTRA}" if EXTRA else "")
!dass {CFG} show-config | head -n 40

In [ ]:
# Tiện ích xem hình / bảng mà các stage đã lưu lên Drive
import glob, json
import pandas as pd
from IPython.display import Image, display

_out = !dass {CFG} show-config
_cfg = json.loads("\n".join(_out))
RESULTS = f"{_cfg['paths']['drive_root']}/results_{_cfg['paths']['run_tag']}"

def show(pattern):
    for p in sorted(glob.glob(f"{RESULTS}/{pattern}")):
        print(p)
        display(Image(p))

def table(name, folder="tables"):
    p = f"{RESULTS}/{folder}/{name}.csv"
    display(pd.read_csv(p)) if os.path.exists(p) else print("chưa có", p)

## 2. Dữ liệu: đọc nguồn, nhận diện số kênh, tiền xử lý, chia train / val / test

In [ ]:
!dass {CFG} prepare
show("class_distribution.png")

## 3. StyleGAN2-ADA có điều kiện

`gan-setup` clone repo NVlabs, vá cho PyTorch 2.x và biên dịch plugin CUDA. `gan` train với early stopping theo KID
của lớp thiểu số — **Colab ngắt thì chạy lại đúng cell này** để resume. GAN đã train xong (ISIC: `checkpoints_v5`,
Brain Tumor: `checkpoints_bt`) được dùng lại ngay. `--fresh-start` XOÁ GAN cũ.

In [ ]:
!dass {CFG} gan-setup

In [ ]:
!dass {CFG} gan
show("gan/*.png")

## 4. Candidate pool

Ảnh sinh lưu đúng số kênh của bộ dữ liệu. Dữ liệu ảnh xám + GAN 3 kênh: chỉ gộp về 1 kênh khi 3 kênh giống hệt
nhau — nếu không, lệnh dừng và yêu cầu train lại GAN 1 kênh.

In [ ]:
!dass {CFG} sample

## 5. (Tuỳ chọn) Fingerprint miền tần số — detector của Frank et al.

In [ ]:
!dass {CFG} fingerprint
table("frequency_fingerprint", "metrics")
show("frequency_analysis/*.png")

## 6. DASS: E_v / E_d, chấm điểm, chọn ảnh cho M0–M6, kiểm tra shortcut

In [ ]:
!dass {CFG} select
table("probe_auc", "metrics")
table("shortcut_check", "metrics")
show("selection_figures/scatter_Mv_Md.png")
show("selection_figures/grid_M6_dass_*.png")

## 7. Train classifier — 6 backbone × 3 seed (`classifier.seeds`)

Mỗi cell một backbone, train trên mọi biến thể M0–M6 với 3 seed; chạy lại được (lần chạy đã có dự đoán trên
Drive sẽ bỏ qua). CNN: `EfficientNetV2B0`, `ResNet50`, `DenseNet121`, `ConvNeXtTiny`; Transformer (KerasHub):
`ViT-B16`, `SwinT`. Transformer nặng hơn CNN nhiều — nên dùng GPU L4 / A100.

Thử nhanh các backbone mới trước khi chạy thật (1 epoch, 1 seed):
`!dass {CFG} -c configs/experiments/smoke.yaml run --from train --to train --models ConvNeXtTiny ViT-B16 SwinT`

In [ ]:
!dass {CFG} train --model EfficientNetV2B0

In [ ]:
!dass {CFG} train --model ResNet50

In [ ]:
!dass {CFG} train --model DenseNet121

In [ ]:
!dass {CFG} train --model ConvNeXtTiny

In [ ]:
!dass {CFG} train --model ViT-B16

In [ ]:
!dass {CFG} train --model SwinT

## 8. Đánh giá và bảng cho bài báo (`tables/*.csv`, `*.json`, `*.tex`)

In [ ]:
!dass {CFG} evaluate
!dass {CFG} report
table("dataset")
table("classification")
table("significance")
table("generation_quality")

## Chạy cả chuỗi bằng một lệnh

```
!dass {CFG} run                         # prepare -> report, train mọi model trong classifier.models
!dass {CFG} run --from select --to report
```